# 第1回 演習: 関数が値であること

スライド: [第1回](../01-basics/)

## 課題のつながり

```
課題1 竹内関数 (とても遅い)
課題2 ディスパッチテーブルで電卓        ─┐
課題3 @memo を作って課題1 に貼る   │  課題1 が一瞬になる
課題4 @command によるコマンド表  ←───────┘  課題2 を文字列から呼べるようにする
```

## 進め方
- 各課題の `TODO` を埋めて、すぐ下の**確認セル**を実行する
- 確認セルが `OK` と出れば正解。`AssertionError` のメッセージに期待値が出る
- 埋め終わったら、**実行結果を残したまま** `.ipynb` を保存して提出

---
## 課題1: 竹内関数

竹内関数 (たらいまわし関数) は、値そのものは単純なのに
**呼び出し回数が爆発する**ことで知られる再帰関数。

```
tarai(x, y, z) = y                                (x <= y のとき)
               = tarai(tarai(x-1, y, z),
                       tarai(y-1, z, x),
                       tarai(z-1, x, y))          (それ以外)
```

実装し、**何回呼ばれたか**を数えよ。

**ヒント**: 関数の中からグローバル変数を書き換えるには `global CALLS` の宣言が要る。

In [ ]:
CALLS = 0


def tarai(x, y, z):
    global CALLS
    CALLS += 1
    # TODO
    pass


import time
CALLS = 0
t0 = time.time()
print('tarai(12, 6, 0) =', tarai(12, 6, 0))
print(f'呼び出し回数: {CALLS:,} 回 / {time.time() - t0:.2f} 秒')

In [ ]:
# 確認
CALLS = 0; assert tarai(1, 2, 3) == 2      # x <= y なので y
CALLS = 0; assert tarai(3, 2, 1) == 3
CALLS = 0
assert tarai(10, 5, 0) == 10
assert CALLS == 343073, f'呼び出し回数が {CALLS} 回。数え方を見直すこと'
print('OK')

`tarai(14, 7, 0)` は呼び出しが5億回を超える。**実行しないこと。**

なぜこんなに増えるのか、下のセルにコメントで書け。
同じ引数の組で何度も呼ばれていないか？

In [ ]:
# 考えたこと:
# 

---
## 課題2: ディスパッチテーブルで電卓

`if` を並べずに、**辞書に関数を入れて**電卓を作る。

- `OPS`: 演算子の文字列から「2引数の関数」を引く辞書
- `calc(op, a, b)`: `OPS` を引いて呼ぶ。**本体は1行**
- 知らない演算子なら `ValueError`

**ヒント**: `operator` モジュールに `add`, `sub`, `mul`, `floordiv`, `mod`, `pow` がある。
`lambda` を自分で書いてもよい。

In [ ]:
import operator

OPS = {
    '+': operator.add,
    # TODO: - * / % ** を追加
}


def calc(op, a, b):
    # TODO: 知らない演算子は ValueError
    pass


print(calc('+', 3, 4), calc('*', 3, 4), calc('**', 2, 10))

In [ ]:
# 確認
assert calc('+', 3, 4) == 7
assert calc('-', 10, 3) == 7
assert calc('*', 3, 4) == 12
assert calc('/', 20, 6) == 3        # 整数除算にすること
assert calc('%', 20, 6) == 2
assert calc('**', 2, 10) == 1024
try:
    calc('?', 1, 2)
except ValueError:
    pass
else:
    raise AssertionError('知らない演算子は ValueError を出すこと')

# 実行中に演算子を増やせること (動的言語らしさ)
OPS['max'] = max
assert calc('max', 3, 9) == 9
print('OK')

---
## 課題3: `@memo` デコレータ

**関数を受け取って、結果をキャッシュする関数を返す**デコレータを書け。

```python
def memo(func):
    cache = {}          # この辞書を wrapper が覚えている (クロージャ)
    def wrapper(*args):
        ...             # cache に無ければ計算して入れる
    return wrapper
```

**ヒント**: `*args` は引数を**タプル**にまとめる。タプルは辞書のキーにできる。

In [ ]:
def memo(func):
    cache = {}

    def wrapper(*args):
        # TODO
        pass

    wrapper.cache = cache      # 中身を外から見られるようにしておく
    return wrapper

In [ ]:
# 確認
calls = 0

@memo
def slow_add(a, b):
    global calls
    calls += 1
    return a + b

assert slow_add(1, 2) == 3
assert slow_add(1, 2) == 3
assert slow_add(1, 2) == 3
assert calls == 1, f'3回呼んだのに本体が {calls} 回実行されている'
assert slow_add(2, 3) == 5
assert calls == 2, '引数が違えば計算し直すこと'
assert len(slow_add.cache) == 2
print('OK')

### 課題3-b: 課題1 に貼る

**`tarai` の中身は1文字も変えずに** `@memo` を付けて、呼び出し回数を比べよ。

In [ ]:
CALLS = 0


@memo
def tarai(x, y, z):
    global CALLS
    CALLS += 1
    # TODO: 課題1 と同じ中身をそのまま書く
    pass


import time
CALLS = 0
t0 = time.time()
print('tarai(12, 6, 0) =', tarai(12, 6, 0))
print(f'呼び出し回数: {CALLS:,} 回 / {time.time() - t0:.4f} 秒')

In [ ]:
# 確認
CALLS = 0
tarai.cache.clear()
assert tarai(12, 6, 0) == 12
assert CALLS == 225, f'{CALLS} 回。@memo が効いていない (課題1 では 12,604,861 回)'

# メモ化すれば、素朴な実装では絶対に終わらない大きさも一瞬で解ける
CALLS = 0
tarai.cache.clear()
assert tarai(100, 50, 0) == 100
print(f'OK  tarai(100, 50, 0) は {CALLS:,} 回で終わった')

---
## 課題4: コマンド表

**定義した瞬間に辞書へ登録する**デコレータを作り、文字列からコマンドを実行できるようにする。

```python
@command('shout')
def shout(text):
    return text.upper() + '!!'

run('shout hello')      # 'HELLO!!'
```

- `command(name)`: **デコレータを返す関数**。`COMMANDS[name]` に登録し、
  **関数そのものはそのまま返す** (包まない)
- `run(line)`: 空白で分け、先頭を命令名、残りを引数として呼ぶ。
  知らない命令なら `ValueError`

**ヒント**: `name, *args = line.split()` で先頭と残りに分けられる。
呼ぶときは `func(*args)`。引数はすべて**文字列**で渡ってくることに注意。

In [ ]:
COMMANDS = {}


def command(name):
    def deco(func):
        # TODO: 登録して、func をそのまま返す
        pass
    return deco


def run(line):
    # TODO
    pass

In [ ]:
# 課題2 の電卓をコマンドとして登録する


@command('calc')
def cmd_calc(op, a, b):
    """2つの整数を演算する"""
    return calc(op, int(a), int(b))


@command('shout')
def cmd_shout(text):
    """大文字にして !! を付ける"""
    return text.upper() + '!!'


@command('rev')
def cmd_rev(text):
    """文字列を逆順にする"""
    return text[::-1]


for line in ['calc * 6 7', 'shout hello', 'rev abc']:
    print(f'{line:16} -> {run(line)}')

In [ ]:
# 確認
assert run('calc * 6 7') == 42
assert run('calc ** 2 10') == 1024
assert run('shout hello') == 'HELLO!!'
assert run('rev abc') == 'cba'
assert set(COMMANDS) == {'calc', 'shout', 'rev'}

# デコレータは関数を包まずそのまま返すこと
assert cmd_shout('hi') == 'HI!!', '@command が関数を壊している'
assert cmd_shout.__name__ == 'cmd_shout'

try:
    run('unknown x')
except ValueError:
    pass
else:
    raise AssertionError('知らない命令は ValueError を出すこと')

# 実行中に命令を増やせること
@command('twice')
def cmd_twice(text):
    """2回くり返す"""
    return text * 2

assert run('twice ab') == 'abab'
print('OK')

---
# 発展課題
余力があれば取り組むこと。

## 発展1: `@trace` で再帰を眺める

呼び出しの深さだけ字下げして、引数と返り値を表示するデコレータを書け。

```
tarai_t(3, 2, 1)
  tarai_t(2, 2, 1)
  -> 2
  ...
-> 3
```

**ヒント**: 深さを覚えるカウンタをクロージャに持たせる。
`try / finally` で、例外が出ても深さが戻るようにするとよい。

**注意**: 大きい引数だと出力が膨大になる。`tarai_t(3, 2, 1)` 程度で試すこと。

In [ ]:
def trace(func):
    depth = 0

    def wrapper(*args):
        nonlocal depth        # 外側の depth を書き換える宣言
        # TODO
        pass

    return wrapper


@trace
def tarai_t(x, y, z):
    if x <= y:
        return y
    return tarai_t(tarai_t(x-1, y, z), tarai_t(y-1, z, x), tarai_t(z-1, x, y))


print('結果:', tarai_t(3, 2, 1))

In [ ]:
# 確認
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    result = tarai_t(3, 2, 1)
out = buf.getvalue()
assert result == 3, '返り値を壊さないこと'
assert out.count('tarai_t') >= 5, '呼び出しが表示されていない'
assert any(l.startswith('  ') for l in out.splitlines()), '深さで字下げすること'
print('OK')

## 発展2: `help` を自動生成する

コマンド表に登録された関数の **docstring と引数名**から、使い方一覧を作る
`help_text()` を書け。

```
calc <op> <a> <b>       2つの整数を演算する
rev <text>              文字列を逆順にする
...
```

**ヒント**: `func.__doc__` で docstring、
`inspect.signature(func).parameters` で引数名の一覧が取れる。コマンド名は辞書順に。

**ここが面白いところ**: 一覧を手で書いていない。
**プログラムが自分自身を調べて**説明を作っている。

In [ ]:
import inspect


def help_text():
    # TODO
    pass


print(help_text())

In [ ]:
# 確認
t = help_text()
for name in COMMANDS:
    assert name in t, f'{name} が一覧に無い'
assert '<op>' in t and '<text>' in t, '引数名を <> で囲んで並べること'
assert '逆順' in t, 'docstring を使うこと'
assert t.index('calc') < t.index('rev') < t.index('shout'), '辞書順に並べること'
print('OK')

---
## 提出

1. すべてのセルを上から順に実行し、出力を残す
2. `ファイル → .ipynb をダウンロード` で保存
3. 締切までに指定の方法で提出

課題1 のあとのコメントも忘れずに埋めること。